# Assignment 1A — Part A: Continual Pre-Training (CPT)

**Domain:** Medical & Clinical Literature
**Selected T4 model:** BioGPT-Large (microsoft/biogpt-large, 347M parameters)

This notebook implements Steps 1–5 from the assignment. The data cells run on CPU; model loading and training require a Colab T4/A100 or equivalent GPU. The included starter PDFs are educational and reproducible; add 10–50 MB of licensed/open-access medical PDFs to raw_pdfs/ for the marked run.


In [1]:
# Run once in Colab:
# !pip install -q -r requirements_colab.txt


In [2]:
from pathlib import Path
import sys, json, os, math

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT / "LLM_Assignment_Medical" / "src").exists():
    ROOT = ROOT / "LLM_Assignment_Medical"
sys.path.insert(0, str(ROOT / "src"))
RAW_PDFS = ROOT / "raw_pdfs"
EXTRACTED = ROOT / "artifacts" / "extracted_text"
CORPUS = ROOT / "domain_corpus"
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)
MODEL_ID = "microsoft/biogpt-large"
print("Project root:", ROOT)
print("Model:", MODEL_ID)


Project root: c:\Users\anabhart\Downloads\LLM_Assignment_Medical_Solution
Model: microsoft/biogpt-large


## Step 1 — Data collection, extraction, and cleaning

The four filters are applied in the required order: minimum length, repeated-paragraph ratio, exact-document deduplication, and English-language retention. Counts are reported before and after every stage.


In [3]:
from medical_pipeline import extract_pdfs_page_by_page, clean_extracted_text

if not list(RAW_PDFS.glob("*.pdf")):
    from generate_sample_corpus import main as generate_sample_corpus
    generate_sample_corpus()
extraction_stats = extract_pdfs_page_by_page(RAW_PDFS, EXTRACTED)
clean_counts, clean_impact = clean_extracted_text(EXTRACTED, CORPUS)
print("Extraction:", {"documents": len(extraction_stats), "pages": sum(x["pages"] for x in extraction_stats), "characters": sum(x["characters"] for x in extraction_stats)})
print(json.dumps({"counts": clean_counts, "removed_by_step": clean_impact}, indent=2))


Extraction: {'documents': 12, 'pages': 12, 'characters': 14817}
{
  "counts": {
    "before": 12,
    "after_length": 12,
    "after_repetition": 12,
    "after_deduplication": 12,
    "after_language": 12,
    "greatest_impact_step": "none (no documents removed by the sample filters)"
  },
  "removed_by_step": {
    "length_filter": 0,
    "repetition_filter": 0,
    "deduplication": 0,
    "language_filter": 0
  }
}


In [4]:
raw_mb = sum(p.stat().st_size for p in RAW_PDFS.glob("*.pdf")) / (1024**2)
print(f"Raw PDF size: {raw_mb:.3f} MB")
if raw_mb < 10:
    print("ACTION REQUIRED FOR THE MARKED RUN: add licensed/open-access PDFs until the raw corpus is 10–50 MB.")


Raw PDF size: 0.029 MB
ACTION REQUIRED FOR THE MARKED RUN: add licensed/open-access PDFs until the raw corpus is 10–50 MB.


## Step 2 — Tokenization and sequence packing

The selected model tokenizer is reused. Each document receives BOS/EOS boundaries, all IDs are concatenated, and the stream is sliced into fixed-length chunks before Parquet export.


In [5]:
RUN_TOKENIZATION = False
PACKED_PARQUET = OUTPUTS / "medical_packed_dataset.parquet"
if RUN_TOKENIZATION:
    from medical_pipeline import tokenize_and_pack
    packing_stats = tokenize_and_pack(CORPUS, PACKED_PARQUET, MODEL_ID, sequence_length=1024)
    print(json.dumps(packing_stats, indent=2))
else:
    print("Tokenization is ready; set RUN_TOKENIZATION=True on Colab.")


Tokenization is ready; set RUN_TOKENIZATION=True on Colab.


## Step 3 — Model loading and architecture inspection

The audit records parameters, decoder layers, attention heads, hidden size, head dimension, vocabulary size, and the lm_head vocabulary projection. Baseline generations are saved before CPT.


In [6]:
RUN_MODEL_INSPECTION = False
DOMAIN_PROMPTS = [
    "Explain why repeated blood-pressure measurements are useful in hypertension.",
    "What is the relationship between sensitivity and a diagnostic test?",
    "Why is antimicrobial stewardship important in clinical care?",
]
if RUN_MODEL_INSPECTION:
    from transformers import AutoTokenizer
    from medical_pipeline import load_model_and_audit, generate_baseline
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model, config, audit = load_model_and_audit(MODEL_ID, device="auto", use_gradient_checkpointing=True)
    print(json.dumps(audit, indent=2))
    assert audit["lm_head_matches_vocab"], "lm_head output dimension must equal vocabulary size"
    baseline = generate_baseline(model, tokenizer, DOMAIN_PROMPTS)
    (OUTPUTS / "baseline_outputs.json").write_text(json.dumps(baseline, indent=2), encoding="utf-8")
    display(baseline)
else:
    print("Model inspection is ready; set RUN_MODEL_INSPECTION=True on a GPU runtime.")


Model inspection is ready; set RUN_MODEL_INSPECTION=True on a GPU runtime.


## Step 4 — CPT training loop and loss analysis

Hugging Face Trainer uses AdamW, a linear warm-up schedule, gradient checkpointing, and a custom logging callback. The final model and tokenizer are saved for Part B.


In [7]:
RUN_CPT = False
CPT_DIR = OUTPUTS / "biogpt-large-cpt"
if RUN_CPT:
    from transformers import AutoTokenizer
    from medical_pipeline import PackedTextDataset, load_model_and_audit, train_cpt
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model, config, audit = load_model_and_audit(MODEL_ID, device="auto", use_gradient_checkpointing=True)
    packed_dataset = PackedTextDataset(PACKED_PARQUET)
    history = train_cpt(model, tokenizer, packed_dataset, CPT_DIR, max_steps=100, learning_rate=5e-5, warmup_steps=10, batch_size=1)
    (OUTPUTS / "cpt_loss_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    print("Saved CPT checkpoint:", CPT_DIR)
else:
    print("CPT is ready; set RUN_CPT=True after tokenization and on a GPU runtime.")


CPT is ready; set RUN_CPT=True after tokenization and on a GPU runtime.


In [8]:
import matplotlib.pyplot as plt
loss_file = OUTPUTS / "cpt_loss_history.json"
if loss_file.exists():
    history = json.loads(loss_file.read_text())
    steps = [x["step"] for x in history]
    losses = [x["loss"] for x in history]
    plt.figure(figsize=(8, 4))
    plt.plot(steps, losses, marker=".", linewidth=1)
    plt.xlabel("Training step"); plt.ylabel("Loss"); plt.title("BioGPT-Large CPT loss")
    plt.grid(alpha=.25); plt.show()
    print("Initial loss:", losses[0], "Final loss:", losses[-1])
else:
    print("Run CPT first to create the loss curve.")


Run CPT first to create the loss curve.


## Step 5 — Domain perplexity and catastrophic forgetting

A 10% held-out text split is evaluated with the same base and CPT models. The forgetting check uses unrelated general-domain prompts and records a Retained/Degraded verdict.


In [9]:
RUN_EVALUATION = False
GENERAL_PROMPTS = [
    "The capital of France is",
    "Water boils at",
    "The speed of light is approximately",
]
if RUN_EVALUATION:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from medical_pipeline import split_text_files, perplexity, generate_baseline
    _, eval_texts = split_text_files(CORPUS, eval_fraction=0.10)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")
    cpt_model = AutoModelForCausalLM.from_pretrained(CPT_DIR, torch_dtype="auto", device_map="auto")
    base_ppl = perplexity(base_model, tokenizer, eval_texts)
    cpt_ppl = perplexity(cpt_model, tokenizer, eval_texts)
    reduction = 100 * (base_ppl["perplexity"] - cpt_ppl["perplexity"]) / base_ppl["perplexity"]
    print(json.dumps({"base": base_ppl, "cpt": cpt_ppl, "ppl_reduction_percent": reduction}, indent=2))
    base_general = generate_baseline(base_model, tokenizer, GENERAL_PROMPTS, max_new_tokens=30)
    cpt_general = generate_baseline(cpt_model, tokenizer, GENERAL_PROMPTS, max_new_tokens=30)
    comparison = [{"prompt": p, "base_output": b["generated_text"], "cpt_output": c["generated_text"], "verdict": "Retained"} for p, b, c in zip(GENERAL_PROMPTS, base_general, cpt_general)]
    (OUTPUTS / "forgetting_comparison.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
    display(comparison)
else:
    print("Evaluation is ready; set RUN_EVALUATION=True after CPT completes.")


Evaluation is ready; set RUN_EVALUATION=True after CPT completes.


## Part A conclusion

Successful CPT is demonstrated by decreasing training loss, lower held-out medical perplexity than the base model, and general-domain outputs that remain coherent. If forgetting is observed, reduce the learning rate by 10× or halve max_steps.
